# Analyse des données énergétiques européennes

Projet M1 IMDS

Objectifs :
- Stocker les données des pays européens dans MongoDB
- Comparer le mix énergétique de plusieurs pays
- Étudier l'évolution de la consommation énergétique en France
- Analyser les corrélations avec la population et d'autres variables
- Créer des visualisations interactives avec Plotly/Dash

In [ ]:
from google.colab import drive
import os
import json

# Monter Google Drive dans un nouveau dossier
drive.mount("/content/gdrive")

# Dossier du projet dans Google Drive
PROJECT_DIR = "/content/gdrive/MyDrive/M1_IMDS_Energy"

# Vérification du contenu
print("Contenu du dossier projet :")
for file in os.listdir(PROJECT_DIR):
    print("-", file)

In [ ]:
JSON_PATH = os.path.join(PROJECT_DIR, "owid-energy-data.json")

with open(JSON_PATH, "r", encoding="utf-8") as f:
    energy_data = json.load(f)

print("Dataset chargé avec succès")
print("Nombre d'entrées :", len(energy_data))

In [ ]:
IMAGES_DIR = os.path.join(PROJECT_DIR, "images")

print("Dossier images :", IMAGES_DIR)

In [ ]:
type(energy_data)

In [ ]:
len(energy_data)

In [ ]:
list(energy_data.keys())[:20]

In [ ]:
energy_data["France"].keys()

In [ ]:
energy_data["France"]["iso_code"]

In [ ]:
len(energy_data["France"]["data"])

In [ ]:
energy_data["France"]["data"][0]

In [ ]:
european_countries = [
    "Albania", "Andorra", "Austria", "Belarus", "Belgium",
    "Bosnia and Herzegovina", "Bulgaria", "Croatia", "Cyprus",
    "Czechia", "Denmark", "Estonia", "Finland", "France",
    "Germany", "Greece", "Hungary", "Iceland", "Ireland",
    "Italy", "Latvia", "Lithuania", "Luxembourg", "Malta",
    "Moldova", "Montenegro", "Netherlands", "North Macedonia",
    "Norway", "Poland", "Portugal", "Romania", "Russia",
    "Serbia", "Slovakia", "Slovenia", "Spain", "Sweden",
    "Switzerland", "Ukraine", "United Kingdom"
]

In [ ]:
available_european_countries = [
    country for country in european_countries
    if country in energy_data
]

len(available_european_countries)

In [ ]:
available_european_countries

In [ ]:
missing_countries = [
    country for country in european_countries
    if country not in energy_data
]

missing_countries

In [ ]:
europe_data = []

for country in available_european_countries:
    document = {
        "country": country,
        "iso_code": energy_data[country].get("iso_code"),
        "data": energy_data[country]["data"]
    }

    europe_data.append(document)

In [ ]:
len(europe_data)

In [ ]:
europe_data[0]

In [ ]:
print(europe_data[0]["country"])
print(europe_data[0]["iso_code"])
print(len(europe_data[0]["data"]))

In [ ]:
!pip install pymongo

In [ ]:
from pymongo import MongoClient
from getpass import getpass
from urllib.parse import quote_plus

username = "naimaassoulaimani80_db_user"
password = getpass("Mot de passe MongoDB : ")

password_encoded = quote_plus(password)

uri = (
    f"mongodb+srv://{username}:{password_encoded}"
    "@energycluster.ei4fvmy.mongodb.net/"
    "?appName=EnergyCluster"
)

client = MongoClient(uri)

In [ ]:
client.admin.command("ping")

In [ ]:
db = client["energy_db"]
collection = db["europe_energy"]

In [ ]:
for document in europe_data:
    collection.update_one(
        {"iso_code": document["iso_code"]},
        {"$set": document},
        upsert=True
    )

In [ ]:
collection.count_documents({})

In [ ]:
collection.find_one(
    {"country": "France"},
    {"_id": 0, "country": 1, "iso_code": 1}
)

In [ ]:
import pandas as pd

france_doc = collection.find_one({"country": "France"})
germany_doc = collection.find_one({"country": "Germany"})

In [ ]:
france_df = pd.DataFrame(france_doc["data"])
germany_df = pd.DataFrame(germany_doc["data"])

In [ ]:
france_df.shape, germany_df.shape

In [ ]:
france_df.head()

In [ ]:
[col for col in france_df.columns if "share_energy" in col]

In [ ]:
[col for col in france_df.columns if "share_elec" in col]

### Comparaison du mix électrique entre la France et l'Allemagne

Dans cette partie, on compare la part des principales sources d'électricité
(charbon, gaz, pétrole, nucléaire, hydraulique, solaire et éolien)
dans les deux pays.

In [ ]:
# Variables utilisées pour comparer le mix électrique
# On garde les principales sources de production d'électricité
energy_sources = [
    "coal_share_elec",
    "gas_share_elec",
    "oil_share_elec",
    "nuclear_share_elec",
    "hydro_share_elec",
    "solar_share_elec",
    "wind_share_elec"
]

In [ ]:
france_df[["year"] + energy_sources].tail(10)

In [ ]:
germany_df[["year"] + energy_sources].tail(10)

### Évolution du mix électrique

On observe ici l'évolution de la part des principales sources de production
d'électricité en France et en Allemagne.

In [ ]:
import matplotlib.pyplot as plt

# On se limite à la période récente pour avoir une comparaison plus lisible
france_recent = france_df[france_df["year"] >= 2000]
germany_recent = germany_df[germany_df["year"] >= 2000]

In [ ]:
# Évolution du mix électrique en France depuis 2000

plt.figure(figsize=(12, 6))

for source in energy_sources:
    plt.plot(
        france_recent["year"],
        france_recent[source],
        label=source.replace("_share_elec", "")
    )

plt.title("Évolution du mix électrique en France depuis 2000")
plt.xlabel("Année")
plt.ylabel("Part dans la production d'électricité (%)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Évolution du mix électrique en Allemagne depuis 2000

plt.figure(figsize=(12, 6))

for source in energy_sources:
    plt.plot(
        germany_recent["year"],
        germany_recent[source],
        label=source.replace("_share_elec", "")
    )

plt.title("Évolution du mix électrique en Allemagne depuis 2000")
plt.xlabel("Année")
plt.ylabel("Part dans la production d'électricité (%)")
plt.legend()
plt.grid(True)
plt.show()

### Interprétation

En France, la production d’électricité reste très largement dominée par le nucléaire sur toute la période, même si sa part diminue légèrement au fil du temps. L’hydraulique reste relativement stable, tandis que l’éolien et le solaire progressent surtout à partir des années 2010.

En Allemagne, le mix électrique évolue davantage. La part du charbon diminue nettement, tandis que l’éolien et le solaire augmentent fortement. La part du nucléaire baisse progressivement jusqu’à devenir presque nulle en 2023.

### Comparaison du mix électrique en 2023

Pour compléter l'analyse temporelle, on compare directement la part des principales
sources d'électricité en France et en Allemagne pour l'année 2023.

In [ ]:
# Récupération des données de l'année 2023

france_2023 = france_df[france_df["year"] == 2023][energy_sources].iloc[0]
germany_2023 = germany_df[germany_df["year"] == 2023][energy_sources].iloc[0]

# Noms plus simples pour l'affichage
labels = [
    source.replace("_share_elec", "")
    for source in energy_sources
]

france_2023, germany_2023

In [ ]:
import numpy as np

# Position des barres sur l'axe horizontal
x = np.arange(len(labels))
width = 0.35

plt.figure(figsize=(12, 6))

plt.bar(
    x - width/2,
    france_2023.values,
    width,
    label="France"
)

plt.bar(
    x + width/2,
    germany_2023.values,
    width,
    label="Allemagne"
)

plt.title("Comparaison du mix électrique France - Allemagne en 2023")
plt.xlabel("Source d'électricité")
plt.ylabel("Part dans la production d'électricité (%)")

plt.xticks(x, labels, rotation=45)
plt.legend()
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

### Interprétation

En 2023, le mix électrique des deux pays est très différent.

La France reste largement dominée par le nucléaire, qui représente environ les deux tiers de la production d'électricité. L'hydraulique et l'éolien occupent une place secondaire, tandis que le charbon et le pétrole sont très faibles.

En Allemagne, le mix est plus diversifié. Le charbon et l'éolien représentent une part importante de la production, suivis du gaz et du solaire. Le nucléaire est devenu très faible en 2023.

Cette comparaison montre donc deux stratégies énergétiques différentes : une forte dépendance au nucléaire en France, contre une place plus importante des énergies renouvelables et des combustibles fossiles en Allemagne.

In [ ]:
# Valeurs du mix électrique en 2023 pour les deux pays

comparison_2023 = pd.DataFrame({
    "Source": labels,
    "France": france_2023.values,
    "Allemagne": germany_2023.values
})

comparison_2023.round(2)

### Comparaison du mix énergétique entre la France et l'Allemagne

Dans cette partie, on compare la part des principales sources d'énergie
dans la consommation énergétique totale des deux pays.

In [ ]:
# Principales sources utilisées pour comparer le mix énergétique global

energy_mix_sources = [
    "coal_share_energy",
    "gas_share_energy",
    "oil_share_energy",
    "nuclear_share_energy",
    "hydro_share_energy",
    "solar_share_energy",
    "wind_share_energy"
]

In [ ]:
# Données du mix énergétique pour l'année 2023

france_energy_2023 = france_df[
    france_df["year"] == 2023
][energy_mix_sources].iloc[0]

germany_energy_2023 = germany_df[
    germany_df["year"] == 2023
][energy_mix_sources].iloc[0]

In [ ]:
energy_labels = [
    source.replace("_share_energy", "")
    for source in energy_mix_sources
]

In [ ]:
# Comparaison du mix énergétique France - Allemagne en 2023

x = np.arange(len(energy_labels))
width = 0.35

plt.figure(figsize=(12, 6))

plt.bar(
    x - width/2,
    france_energy_2023.values,
    width,
    label="France"
)

plt.bar(
    x + width/2,
    germany_energy_2023.values,
    width,
    label="Allemagne"
)

plt.title("Comparaison du mix énergétique France - Allemagne en 2023")
plt.xlabel("Source d'énergie")
plt.ylabel("Part dans la consommation énergétique totale (%)")

plt.xticks(x, energy_labels, rotation=45)
plt.legend()
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(
    os.path.join(IMAGES_DIR, "mix_energetique_2023.png"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()

### Interprétation

En 2023, le mix énergétique de la France et de l'Allemagne présente des différences importantes.

En France, le nucléaire occupe une place très importante dans la consommation énergétique totale,
avec une part d'environ 35 %. Le pétrole représente également une part élevée, autour de 32 %.

En Allemagne, le pétrole reste la principale source d'énergie, suivi du gaz et du charbon.
La part du nucléaire est en revanche très faible.

Les énergies renouvelables comme l'éolien et le solaire sont présentes dans les deux pays,
mais leur part reste plus faible que celle des principales sources fossiles ou du nucléaire.

Cette comparaison montre donc que la France repose davantage sur le nucléaire, tandis que l'Allemagne dépend davantage des combustibles fossiles comme le pétrole, le gaz et le charbon.

### Évolution de la consommation énergétique en France

Dans cette partie, on étudie l'évolution de la consommation énergétique française
selon les principales sources d'énergie.

In [ ]:
# Recherche des variables liées à la consommation énergétique

[col for col in france_df.columns if "consumption" in col]

In [ ]:
# Variables de consommation pour les principales sources d'énergie

consumption_sources = [
    "coal_consumption",
    "gas_consumption",
    "oil_consumption",
    "nuclear_consumption",
    "hydro_consumption",
    "solar_consumption",
    "wind_consumption",
    "biofuel_consumption"
]

france_df[["year"] + consumption_sources].tail(10)

### Évolution de la consommation énergétique par source

On représente ici l'évolution de la consommation des principales sources d'énergie
en France afin d'identifier les sources qui diminuent ou progressent au cours du temps.

In [ ]:
# On garde les données à partir de 2000 pour avoir un graphique plus lisible

france_consumption_recent = france_df[france_df["year"] >= 2000]

plt.figure(figsize=(12, 6))

for source in consumption_sources:
    plt.plot(
        france_consumption_recent["year"],
        france_consumption_recent[source],
        label=source.replace("_consumption", "")
    )

plt.title("Évolution de la consommation énergétique en France depuis 2000")
plt.xlabel("Année")
plt.ylabel("Consommation énergétique (TWh)")
plt.legend()
plt.grid(True)

plt.show()

### Interprétation

Depuis 2000, la consommation énergétique en France évolue différemment selon les sources.

Le nucléaire reste la source la plus importante sur toute la période, même si sa consommation diminue globalement, avec une baisse particulièrement visible autour de 2022 avant une remontée en 2023.

Le pétrole reste également très important, mais sa consommation diminue progressivement au fil des années.

Le gaz est plus stable, avec quelques variations, tandis que le charbon diminue fortement et devient beaucoup moins important en fin de période.

À l'inverse, les consommations liées au solaire et à l'éolien augmentent progressivement, surtout à partir des années 2010. Cela montre une progression des énergies renouvelables dans le mix énergétique français.

### Analyse des corrélations

Dans cette partie, on cherche à voir s'il existe une relation entre la consommation énergétique en France et certaines variables comme la population ou le PIB.

In [ ]:
# Variable de consommation énergétique totale disponible dans le dataset

france_df["total_energy_consumption"] = france_df["primary_energy_consumption"]

In [ ]:
correlation_data = france_df[
    ["year", "population", "gdp", "total_energy_consumption"]
].dropna()

In [ ]:
# Vérification de la période utilisée pour le calcul des corrélations

print("Première année :", correlation_data["year"].min())
print("Dernière année :", correlation_data["year"].max())
print("Nombre d'observations :", len(correlation_data))

In [ ]:
# Vérification des valeurs manquantes pour les variables utilisées

france_df[
    ["year", "population", "gdp", "total_energy_consumption"]
].isna().sum()

In [ ]:
# Années où au moins une variable est manquante

missing_rows = france_df[
    ["year", "population", "gdp", "total_energy_consumption"]
][
    france_df[
        ["population", "gdp", "total_energy_consumption"]
    ].isna().any(axis=1)
]

missing_rows

**Remarque :** l'analyse des corrélations est réalisée sur la période 1965–2022,
car ce sont les années pour lesquelles la population, le PIB et la consommation
énergétique totale sont disponibles simultanément.

In [ ]:
# Matrice de corrélation entre consommation, population et PIB

correlation_matrix = correlation_data[
    ["population", "gdp", "total_energy_consumption"]
].corr()

correlation_matrix

In [ ]:
# Visualisation de la matrice de corrélation

plt.figure(figsize=(7, 5))

plt.imshow(correlation_matrix, cmap="coolwarm", vmin=-1, vmax=1)

plt.colorbar(label="Coefficient de corrélation")

plt.xticks(
    range(len(correlation_matrix.columns)),
    correlation_matrix.columns,
    rotation=45
)

plt.yticks(
    range(len(correlation_matrix.index)),
    correlation_matrix.index
)

for i in range(len(correlation_matrix.index)):
    for j in range(len(correlation_matrix.columns)):
        plt.text(
            j,
            i,
            f"{correlation_matrix.iloc[i, j]:.2f}",
            ha="center",
            va="center"
        )

plt.title("Matrice de corrélation")
plt.tight_layout()
plt.savefig(
    os.path.join(IMAGES_DIR, "correlation_matrix.png"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# Relation entre population et consommation énergétique totale

plt.figure(figsize=(7, 5))
plt.scatter(
    correlation_data["population"],
    correlation_data["total_energy_consumption"]
)

plt.title("Population et consommation énergétique en France")
plt.xlabel("Population")
plt.ylabel("Consommation énergétique totale (TWh)")
plt.grid(True)

plt.show()

In [ ]:
# Relation entre PIB et consommation énergétique totale

plt.figure(figsize=(7, 5))
plt.scatter(
    correlation_data["gdp"],
    correlation_data["total_energy_consumption"]
)

plt.title("PIB et consommation énergétique en France")
plt.xlabel("PIB")
plt.ylabel("Consommation énergétique totale (TWh)")
plt.grid(True)

plt.show()

### Interprétation des corrélations

La population et le PIB présentent une très forte corrélation positive (0,99).

La consommation énergétique totale est également positivement corrélée avec la population (0,77)
et avec le PIB (0,79). Cela montre que, sur la période 1965–2022, ces variables ont globalement
évolué dans le même sens.

Cependant, les nuages de points montrent que cette relation n'est pas parfaitement linéaire.
Pour les valeurs les plus élevées de population et de PIB, on observe même une baisse de la
consommation énergétique totale.

Ces résultats doivent donc être interprétés avec prudence. D'autres facteurs, comme
l'amélioration de l'efficacité énergétique, les changements technologiques ou l'évolution
du mix énergétique, peuvent également influencer la consommation.

Enfin, une corrélation ne permet pas d'établir une relation de causalité.

### Dashboard interactif

Dans cette partie, on crée des visualisations interactives avec Plotly et Dash
pour explorer le mix électrique de la France et de l'Allemagne et suivre l'évolution
des différentes sources d'électricité dans le temps.

In [ ]:
!pip install dash plotly

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

from dash import Dash, dcc, html, Input, Output

In [ ]:
# Préparation des données pour Plotly

mix_2023 = pd.DataFrame({
    "Source": labels,
    "France": france_2023.values,
    "Allemagne": germany_2023.values
})

mix_2023

In [ ]:
# Transformation du tableau pour faciliter l'affichage avec Plotly

mix_2023_long = mix_2023.melt(
    id_vars="Source",
    var_name="Pays",
    value_name="Part"
)

mix_2023_long.head()

In [ ]:
# Graphique interactif du mix électrique en 2023

fig = px.bar(
    mix_2023_long,
    x="Source",
    y="Part",
    color="Pays",
    barmode="group",
    title="Comparaison du mix électrique France - Allemagne en 2023",
    labels={"Part": "Part dans la production d'électricité (%)"}
)

fig.show()

### Dashboard interactif avec Dash

Ce dashboard permet de choisir un pays et une source d'énergie afin de visualiser
leur évolution dans le temps.

In [ ]:
# Création de l'application Dash

app = Dash(__name__)

app.layout = html.Div([

    html.H2("Dashboard énergétique - France et Allemagne"),

    html.Label("Choisir un pays :"),

    dcc.Dropdown(
        id="country-dropdown",
        options=[
            {"label": "France", "value": "France"},
            {"label": "Allemagne", "value": "Germany"}
        ],
        value="France"
    ),

    html.Br(),

    html.Label("Choisir une source d'énergie :"),

    dcc.Dropdown(
        id="source-dropdown",
        options=[
            {"label": "Charbon", "value": "coal_share_elec"},
            {"label": "Gaz", "value": "gas_share_elec"},
            {"label": "Pétrole", "value": "oil_share_elec"},
            {"label": "Nucléaire", "value": "nuclear_share_elec"},
            {"label": "Hydraulique", "value": "hydro_share_elec"},
            {"label": "Solaire", "value": "solar_share_elec"},
            {"label": "Éolien", "value": "wind_share_elec"}
        ],
        value="nuclear_share_elec"
    ),

    dcc.Graph(id="energy-graph")
])

In [ ]:
# Noms des sources pour avoir un affichage plus lisible

source_labels = {
    "coal_share_elec": "Charbon",
    "gas_share_elec": "Gaz",
    "oil_share_elec": "Pétrole",
    "nuclear_share_elec": "Nucléaire",
    "hydro_share_elec": "Hydraulique",
    "solar_share_elec": "Solaire",
    "wind_share_elec": "Éolien"
}

In [ ]:
# Mise à jour du graphique selon le pays et la source choisis

@app.callback(
    Output("energy-graph", "figure"),
    Input("country-dropdown", "value"),
    Input("source-dropdown", "value")
)
def update_graph(country, source):

    if country == "France":
        df = france_recent
        country_name = "France"
    else:
        df = germany_recent
        country_name = "Allemagne"

    source_name = source_labels[source]

    fig = px.line(
        df,
        x="year",
        y=source,
        markers=True,
        title=f"Évolution de la part du {source_name.lower()} en {country_name}",
        labels={
            "year": "Année",
            source: "Part dans la production d'électricité (%)"
        }
    )

    return fig

In [ ]:
app.run(debug=False)

## Conclusion générale

Ce projet a permis d'analyser les données énergétiques de plusieurs pays européens à partir du jeu de données Our World in Data.

Dans un premier temps, les données ont été explorées et filtrées afin de conserver uniquement les pays européens. Elles ont ensuite été stockées dans MongoDB Atlas avec un document par pays, ce qui a permis de structurer les données et de les interroger facilement depuis Python.

L'analyse comparative entre la France et l'Allemagne met en évidence des différences importantes aussi bien dans leur mix électrique que dans leur mix énergétique global. La France se distingue par une place très importante du nucléaire, tandis que l'Allemagne présente une part plus élevée de charbon, de gaz, de pétrole et d'énergies renouvelables comme l'éolien et le solaire.

L'étude de l'évolution de la consommation énergétique en France depuis 2000 montre également des tendances différentes selon les sources. La consommation de charbon diminue fortement, tandis que le pétrole et le nucléaire restent importants mais présentent une baisse globale sur la période. À l'inverse, les consommations liées au solaire et à l'éolien progressent progressivement.

L'analyse des corrélations, réalisée sur la période 1965–2022, montre une relation positive entre la consommation énergétique totale et la population (0,77), ainsi qu'entre la consommation énergétique totale et le PIB (0,79). Cependant, les nuages de points montrent que ces relations ne sont pas parfaitement linéaires. D'autres facteurs, comme l'amélioration de l'efficacité énergétique, les changements technologiques ou l'évolution du mix énergétique, peuvent également avoir un impact sur la consommation. Il est donc important de rappeler qu'une corrélation ne signifie pas nécessairement une relation de causalité.

Enfin, la création d'un dashboard interactif avec Plotly et Dash permet de visualiser l'évolution des différentes sources d'électricité en France et en Allemagne de manière plus dynamique.

Ce projet a donc permis de mettre en pratique plusieurs compétences en analyse de données, notamment la manipulation de fichiers JSON, l'utilisation de MongoDB, l'analyse avec Pandas, la visualisation avec Matplotlib et Plotly, ainsi que la création d'un dashboard interactif avec Dash.